# Chapter 6: The spectral first-return solver, step by step

*Code cells: MIT licence (`LICENSE`). Text and figures: CC BY 4.0 (`DATA_LICENSE.md`). Part of the code-and-data repository for "Mean span dimension of an ideal wormlike chain" (Y. Wang, 2026).*

In [1]:
import sys, json, math, time
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'tutorial' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'code'))
import numpy as np, pandas as pd, sympy as sp
import matplotlib.pyplot as plt
pd.set_option('display.precision', 8)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})
REF = pd.read_csv(ROOT / 'data' / 'wlc_mean_span_reference.csv')
LAM, F = REF['lambda'].to_numpy(), REF['F_continuum'].to_numpy()
Z_M, B_COEF, A_GAUSS = 0.71821164, 0.64680, 4/math.sqrt(3*math.pi)
def check(name, value, expected, tol):
    ok = abs(value - expected) <= tol
    print(('PASS' if ok else 'FAIL') + f'  {name}: {value:.10g}  (expected {expected:.10g}, tol {tol:g})')
    return ok

## The problem to solve

$q(y,u;\lambda)$ obeys $\partial_\lambda q+u\,\partial_yq=\mathcal L_uq$ with $q(0^+,u>0)=1$, $q(\cdot,\cdot;0)=0$ and
$q\to0$ as $y\to\infty$; $F=\int_0^\infty\!\int_{-1}^1q\,du\,dy$. We build the solver of `code/wlc_pn_transfer.py`
one step at a time and then run it.

### Step 1: Laplace transform in $\lambda$

$u\,\partial_y\widehat q=\mathcal L_u\widehat q-s\widehat q$ with $\widehat q(0^+,u>0)=1/s$. The time variable is gone.

### Step 2: angular discretization

In the orthonormal Legendre basis $\phi_\ell$, $\mathcal L_u$ is diagonal ($-\ell(\ell+1)/2$) and $u\cdot$ is
tridiagonal. Evaluating at the $m$ Gauss-Legendre nodes gives $\partial_y\mathbf g=A_m(s)\mathbf g$,
$A_m=U^{-1}(R_m-sI)$, $R_m=-V^T\Lambda_mV$.

In [2]:
from wlc_pn_transfer import angular_system, laplace_transform, invert_euler_bromwich, BromwichSettings, telegraph_p1_transform
sysm = angular_system(8)
print('nodes u_k  :', np.round(sysm.nodes, 4))
print('Lambda diag:', np.diag(sysm.Lambda))
J = sysm.V @ np.diag(sysm.nodes) @ sysm.V.T
print('streaming matrix J (moment space) is tridiagonal:', np.allclose(J - np.diag(np.diag(J)) - np.diag(np.diag(J,1),1) - np.diag(np.diag(J,-1),-1), 0))
print('(J)_{l,l+1} =', np.round(np.diag(J,1), 5), ' vs (l+1)/sqrt((2l+1)(2l+3)) =', np.round([(l+1)/math.sqrt((2*l+1)*(2*l+3)) for l in range(7)], 5))

nodes u_k  : [-0.9603 -0.7967 -0.5255 -0.1834  0.1834  0.5255  0.7967  0.9603]
Lambda diag: [ 0.  1.  3.  6. 10. 15. 21. 28.]
streaming matrix J (moment space) is tridiagonal: True
(J)_{l,l+1} = [0.57735 0.5164  0.50709 0.50395 0.50252 0.50175 0.50128]  vs (l+1)/sqrt((2l+1)(2l+3)) = [0.57735 0.5164  0.50709 0.50395 0.50252 0.50175 0.50128]


### Step 3: the half-space boundary-value problem

The decaying solutions span the stable invariant subspace of $A_m(s)$. Its dimension must equal the
number of incoming nodes, $m/2$, so that the incoming values determine the solution. The code checks
this at every transform point. The outgoing values at $y=0$ (the return-time distribution) are then an
*output*. Spatial integration is analytic: $\int_0^\infty\mathbf g\,dy=-A_m^{-1}\mathbf g(0)$.

The production solver (`build_reference_curve.METHOD = "schur"`) obtains an orthonormal basis of the
stable subspace from an ordered complex Schur factorization; `data/conditioning_audit_full.csv` records
the condition number of $E_+Z_-$ in that basis at orders 8 to 128 and at the 40 transform points of the
primary inversion for $\lambda=0.01$ to $10^5$ (maximum 1.45). A plain eigendecomposition
(`method="eig"`) spans the same subspace with a non-orthogonal basis; it reproduces the Schur transform
to $10^{-13}$ and the reference values to $1.3\times10^{-10}$ and is kept as a regression check
(`code/solver_regression_audit.py`).

In [3]:
s = 0.7 + 0.3j
for m in (8, 16, 32):
    A = (angular_system(m).R.astype(complex) - s*np.eye(m)) / angular_system(m).nodes[:, None]
    ev = np.linalg.eigvals(A)
    print(f'm={m:3d}: eigenvalues with Re<0: {int((ev.real<0).sum())}  (incoming nodes: {m//2})')
val, diag = laplace_transform(s, 32, diagnostics=True)
print('\nF_hat_32(s) =', val, '   stable dim', diag.stable_dimension, '  cond(E+ Z-) =', round(diag.incoming_condition_number, 4))
val_eig = laplace_transform(s, 32, method='eig')
print('eigendecomposition route (regression check): |difference| =', abs(val - val_eig))
cond = pd.read_csv(ROOT/'data'/'conditioning_audit_full.csv')
print('conditioning audit: orders', sorted(cond.m.unique()), ' max cond(E+ Z-) =', cond.condition_max.max() if 'condition_max' in cond else cond.iloc[:, -1].max())

m=  8: eigenvalues with Re<0: 4  (incoming nodes: 4)
m= 16: eigenvalues with Re<0: 8  (incoming nodes: 8)
m= 32: eigenvalues with Re<0: 16  (incoming nodes: 16)

F_hat_32(s) = (0.5242591234380729-0.48778525130986594j)    stable dim 16   cond(E+ Z-) = 1.1
eigendecomposition route (regression check): |difference| = 5.978733960281817e-16
conditioning audit: orders [np.int64(8), np.int64(16), np.int64(24), np.int64(32), np.int64(48), np.int64(64), np.int64(96), np.int64(128)]  max cond(E+ Z-) = 1.4602058412902457


### Step 3 in formulas

1. **Nodal variables.** Write $\widehat q(y,u)=\sum_{\ell<m}a_\ell(y)\phi_\ell(u)$ and $g_k=\sqrt{w_k}\,\widehat q(y,u_k)$,
   i.e. $\mathbf g=V^T\mathbf a$ with $V_{\ell k}=\sqrt{w_k}\phi_\ell(u_k)$. $V$ is orthogonal because the $m$-point Gauss
   rule integrates products $\phi_\ell\phi_{\ell'}$ ($\ell,\ell'<m$) exactly.
2. **The system.** $u\,\partial_y\widehat q=(\mathcal L_u-s)\widehat q$ becomes $D_u\partial_y\mathbf g=(R_m-sI)\mathbf g$ with
   $R_m=-V^T\Lambda_mV$, hence $\partial_y\mathbf g=A_m\mathbf g$, $A_m=D_u^{-1}(R_m-sI)$.
3. **Decay.** A solution that vanishes as $y\to\infty$ is $\mathbf g(y)=Z_-e^{y\,T_-}\mathbf c$, where the columns of
   $Z_-$ span the stable invariant subspace of $A_m$ and $T_-$ is $A_m$ restricted to it.
4. **Boundary condition.** On the incoming nodes $u_k>0$, $\widehat q=1/s$, i.e. $g_k=\sqrt{w_k}/s$:
   $E_+Z_-\mathbf c=\mathbf h_+/s$. This is a square system when the stable subspace has dimension $m/2$.
5. **Integration.** $\int_0^\infty\mathbf g\,dy=-A_m^{-1}\mathbf g(0)$ for a decaying solution, and
   $\int_{-1}^1\widehat q\,du=\sum_kw_k\widehat q(u_k)=\mathbf h^T\mathbf g$ with $h_k=\sqrt{w_k}$. Therefore
   $\widehat F_m(s)=\int_0^\infty\!\int_{-1}^1\widehat q\,du\,dy=-\tfrac1s\,\mathbf h^TA_m^{-1}Z_-(E_+Z_-)^{-1}\mathbf h_+$.

The cell repeats these five steps with plain eigenvectors at $m=8$, integrates each decaying mode
separately, and compares with the production routine.

In [4]:
m_, s_ = 8, 0.7 + 0.3j
sy = angular_system(m_); un = sy.nodes
wk = np.polynomial.legendre.leggauss(m_)[1]
print('V is orthogonal:', np.allclose(sy.V @ sy.V.T, np.eye(m_)))
A = (sy.R.astype(complex) - s_*np.eye(m_))/un[:, None]
mu, vec = np.linalg.eig(A)
dec = mu.real < 0                                                   # decaying modes
c_ = np.linalg.solve(vec[np.ix_(un > 0, dec)], np.sqrt(wk[un > 0])/s_)   # boundary condition on incoming nodes
F_modes = sum(c_[j]*(np.sqrt(wk) @ vec[:, dec][:, j])/(-mu[dec][j]) for j in range(dec.sum()))   # mode-by-mode integral
print('mode-by-mode integral :', F_modes)
print('production routine    :', laplace_transform(s_, m_))
print('outgoing trace s*q(0,-v) at the outgoing nodes (an output of the solve, between 0 and 1 for real s):')
s_r = 0.7; A_r = (sy.R - s_r*np.eye(m_))/un[:, None]; mu_r, vec_r = np.linalg.eig(A_r); dr = mu_r.real < 0
g0 = vec_r[:, dr] @ np.linalg.solve(vec_r[np.ix_(un > 0, dr)], np.sqrt(wk[un > 0])/s_r)
print('   v =', np.round(-un[un < 0], 3), '\n   s*q =', np.round((s_r*g0/np.sqrt(wk)).real[un < 0], 4))

V is orthogonal: True
mode-by-mode integral : (0.5256057174639313-0.4886638708988894j)
production routine    : (0.5256057174639306-0.4886638708988892j)
outgoing trace s*q(0,-v) at the outgoing nodes (an output of the solve, between 0 and 1 for real s):
   v = [0.96  0.797 0.526 0.183] 
   s*q = [0.0784 0.1192 0.2189 0.4966]


### Step 4: inversion and continuum extrapolation

The Euler-accelerated Fourier series (Abate and Whitt 1992) inverts $\widehat F_m$ with $N+K+1$ evaluations
at $s_j=(A_E+2\pi ij)/(2\lambda)$. The ideal series equals $\sum_{j\ge0}e^{-jA_E}F((2j+1)\lambda)$, so the
combination $F_A(\lambda)-e^{-A_E}F_A(3\lambda)$ removes the first alias (`invert_alias_corrected`). This allows
the small parameter $A_E=12$ (with $N=K=32$), which matters because the prefactor $e^{A_E/2}$ multiplies
floating-point noise in $\widehat F_m$: with the uncorrected $A_E=22$ of an earlier version of the table, that
noise grew from a few $10^{-8}$ relative at $\lambda=10^3$ to nearly $10^{-6}$ for $\lambda\ge5\times10^4$
(`validation/inversion_noise_audit.csv`). Each angular
order gives $F_m(\lambda)$; the orders 32, 48, 64, 96 are extrapolated with $F_\infty+c_3m^{-3}+c_4m^{-4}$, and a
fifth order, $m=128$, enters only the sensitivity. The algorithm box of the supplementary material lists the whole procedure. We run
it live at $\lambda=1$ and compare with the reference table.

In [5]:
from build_reference_curve import evaluate_node, continuum_fit
t0 = time.time(); row = evaluate_node(1.0); dt = time.time() - t0
ref1 = REF[REF['lambda'] == 1.0].iloc[0]
print(f'{dt:.1f} s for one node on one core')
for k in ('F_m32','F_m48','F_m64','F_m96','F_continuum','epsilon_internal_rel'):
    print(f'{k:22s} live {row[k]:.10g}    archived {ref1[k]:.10g}')
check('live F_continuum(1) vs archive', row['F_continuum'], ref1['F_continuum'], 1e-9)

7.0 s for one node on one core
F_m32                  live 0.4569267499    archived 0.4569267499
F_m48                  live 0.4569117598    archived 0.4569117598
F_m64                  live 0.4569080683    archived 0.4569080683
F_m96                  live 0.456906158    archived 0.456906158
F_continuum            live 0.4569053478    archived 0.4569053478
epsilon_internal_rel   live 1.773166657e-06    archived 1.773166657e-06
PASS  live F_continuum(1) vs archive: 0.4569053478  (expected 0.4569053478, tol 1e-09)


np.True_

### Where the alias series comes from, and the bound on what remains

The Fourier-series method replaces the Bromwich integral by a trapezoidal sum with step $\pi/\lambda$ in
$\Im s$. By the Poisson summation formula, a trapezoidal sum of a Fourier integral returns the function
plus its periodic copies, here damped by the contour shift $A_E/(2\lambda)$:
$$F_A(\lambda)=\sum_{j\ge0}e^{-jA_E}F\big((2j+1)\lambda\big)=F(\lambda)+e^{-A_E}F(3\lambda)+e^{-2A_E}F(5\lambda)+\dots$$
Apply the same series at $3\lambda$ and subtract $e^{-A_E}$ times it:
$$F_A(\lambda)-e^{-A_E}F_A(3\lambda)=F(\lambda)-e^{-2A_E}\big[F(9\lambda)-F(5\lambda)\big]+O(e^{-3A_E}).$$
The remaining alias is negative because $F$ increases. Subadditivity (Chapter 2) bounds it:
$F(9\lambda)\le F(5\lambda)+F(4\lambda)\le F(5\lambda)+4F(\lambda)$, so its magnitude is below $4e^{-2A_E}F(\lambda)$,
which is $1.5\times10^{-10}F$ at $A_E=12$.

The cell shows both statements on the two-mode ($m=2$) problem, whose exact inverse is the telegraph
range $F_T$ of Chapter 5: the uncorrected series is off by $e^{-A_E}F_T(3\lambda)$, and the corrected one by the
predicted remainder.

In [6]:
from scipy.special import ive
from wlc_pn_transfer import invert_alias_corrected
c_t = 1/math.sqrt(3)
FT2 = lambda t: 2*c_t*(ive(0, t/2) + t*(ive(0, t/2) + ive(1, t/2)) - 1)       # telegraph range with a = 1
st = BromwichSettings(12.0, 32, 32); AE = st.A
print(' lambda   uncorrected error    e^-A F(3 lam) + e^-2A F(5 lam)   corrected error     -e^-2A [F(9 lam) - F(5 lam)]    bound 4 e^-2A F')
for lam_ in (0.1, 1.0, 10.0, 1e3):
    raw = invert_euler_bromwich(lam_, 2, st) - FT2(lam_)
    cor = invert_alias_corrected(lam_, 2, st) - FT2(lam_)
    print(f'{lam_:7g}   {raw:+.6e}        {math.exp(-AE)*FT2(3*lam_) + math.exp(-2*AE)*FT2(5*lam_):+.6e}              '
          f'{cor:+.3e}          {-math.exp(-2*AE)*(FT2(9*lam_) - FT2(5*lam_)):+.3e}                 {4*math.exp(-2*AE)*FT2(lam_):.3e}')

 lambda   uncorrected error    e^-A F(3 lam) + e^-2A F(5 lam)   corrected error     -e^-2A [F(9 lam) - F(5 lam)]    bound 4 e^-2A F
    0.1   +1.026217e-06        +1.026217e-06              -7.430e-12          -7.430e-12                 8.611e-12
      1   +7.994782e-06        +7.994782e-06              -3.607e-11          -3.607e-11                 7.785e-11
     10   +3.712069e-05        +3.712069e-05              -1.184e-10          -1.184e-10                 4.636e-10
   1000   +4.314267e-04        +4.314267e-04              -1.194e-09          -1.188e-09                 6.049e-09


### Convergence in angular order

The finite-order error falls as $m^{-3}$ (the slope of the log-log plot). The article attributes this to the
corner singularity of $q$ at the grazing point $(y,u)=(0,0)$, where the prescribed and free halves of the
boundary meet.

In [7]:
ms = np.array([32, 48, 64, 96]); vals = np.array([row[f'F_m{m}'] for m in ms])
err = np.abs(vals/row['F_continuum'] - 1)
slope = np.polyfit(np.log(ms), np.log(err), 1)[0]
print('relative errors:', err, '\nlog-log slope  :', round(slope, 2), ' (m^-3 expected)')
# withheld orders 128, 192, 256 from the archived spot check (code/high_order_spot_checks.py):
# an m^-3 + m^-4 fit to these three orders alone is an extrapolation independent of the one in the table
spot = pd.read_csv(ROOT/'data'/'high_order_spot_checks.csv')
cols = ['lambda', 'relative_difference_m128_pct', 'relative_difference_extrapolation_pct',
        'scaled_residual_m128', 'scaled_residual_m192', 'scaled_residual_m256']
print(spot[cols].to_string(index=False))
print('\n(F_m - F_inf) m^3 is constant within 2.5 % for 0.1 <= lambda <= 100: the m^-3 law holds there.')
print('At lambda = 1000 the high-order differences are at the noise level (about 1e-9 relative).')
print('At lambda = 0.01 the m^-3 regime is not reached at m <= 256, which is why the short-chain diagnostics go to m = 512.')
ver = pd.read_csv(ROOT/'validation'/'inversion_noise_audit.csv')
print('\nDeviation from the long-chain expansion for lambda >= 5000, former A=22 table vs current table:')
print(ver[ver['lambda'] >= 5000][['lambda', 'dev_longchain_schur_A22', 'dev_longchain_current', 'eps_rel_current']].to_string(index=False))

relative errors: [4.68414707e-05 1.40335479e-05 5.95411225e-06 1.77316666e-06] 
log-log slope  : -2.98  (m^-3 expected)
 lambda  relative_difference_m128_pct  relative_difference_extrapolation_pct  scaled_residual_m128  scaled_residual_m192  scaled_residual_m256
   0.01                    0.00086199                         1.37732208e-04            0.09029937            0.12418275            0.19072190
   0.10                    0.00022587                        -5.38660216e-07            0.23458715            0.23361107            0.23119994
   1.00                    0.00007488                        -2.93685065e-07            0.71748958            0.71319929            0.70138071
  10.00                    0.00002758                        -8.74034956e-08            1.66939901            1.66329309            1.64205319
 100.00                    0.00000903                         3.77899934e-09            2.20823152            2.20299914            2.20355877
1000.00               

### An exactly solvable test of the inversion

At $m=2$ the transfer problem is a telegraph process with speed $1/\sqrt3$ and decay rate 1, whose transform
is known in closed form (`telegraph_p1_transform`). Comparing the numerical transform with it isolates
the linear-algebra part; inverting it with the production series tests the Fourier inversion against the
Bessel-function range. The remaining alias of the corrected series is $-e^{-2A_E}[F(9\lambda)-F(5\lambda)]$,
negative and below $4e^{-2A_E}F$ by subadditivity; `code/telegraph_inversion_test.py` checks this at all 67
lengths (`validation/telegraph_inversion_test.csv`).

In [8]:
for s in (0.5, 1+2j, 5-0.5j):
    print(f's={s!s:12}  numerical {laplace_transform(s, 2):.12g}   analytic {telegraph_p1_transform(s):.12g}')
from scipy.special import ive
c, a = 1/math.sqrt(3), 1.0
FT = lambda t: (2*c/a)*(ive(0, a*t/2) + a*t*(ive(0, a*t/2) + ive(1, a*t/2)) - 1)
from wlc_pn_transfer import invert_alias_corrected
from build_reference_curve import PRIMARY
for lam in (0.1, 10.0, 1e3, 1e5):
    num = invert_alias_corrected(lam, 2, PRIMARY)
    pred = -math.exp(-2*PRIMARY.A)*(FT(9*lam) - FT(5*lam))
    print(f'lambda={lam:8g}  relative error {num/FT(lam)-1:+.3e}   predicted remaining alias {pred/FT(lam):+.3e}')
tel = pd.read_csv(ROOT/'validation'/'telegraph_inversion_test.csv')
print('\nall 67 lengths, analytic transform: errors from %.2e to %.2e; max |error - predicted| = %.1e' % (
      tel.rel_error_analytic_transform.min(), tel.rel_error_analytic_transform.max(),
      (tel.rel_error_analytic_transform - tel.predicted_rel_residual_alias).abs().max()))

s=0.5           numerical 1.69059892324+0j   analytic 1.69059892324+0j
s=(1+2j)        numerical -0.0577692617587-0.0924856162153j   analytic -0.0577692617587-0.0924856162153j
s=(5-0.5j)      numerical 0.0214188345574+0.00423084844816j   analytic 0.0214188345574+0.00423084844816j
lambda=     0.1  relative error -1.303e-10   predicted remaining alias -1.303e-10
lambda=      10  relative error -3.856e-11   predicted remaining alias -3.856e-11
lambda=    1000  relative error -2.979e-11   predicted remaining alias -2.966e-11
lambda=  100000  relative error -2.493e-11   predicted remaining alias -2.892e-11

all 67 lengths, analytic transform: errors from -1.49e-10 to -2.89e-11; max |error - predicted| = 1.1e-14


### Cost and what limits the accuracy

One node costs seconds, so the limit is accuracy, not cost, and it has one cause: the full-range Legendre
basis resolves the grazing-point singularity only algebraically. Two improvements are possible but
are not implemented here: a half-range (double-$P_N$) basis, or subtraction of the known local
random-acceleration solution at the corner, after which the remainder should converge exponentially. The
stationary Milne problem of Chapter 5, with its known $z_M$, is the natural test bed.